# Step 3: LLM on Kaggle
Same files as the project, plus Ollama. Settings: GPU T4 x2, Internet on.

In [ ]:
!pip install -q openai tqdm
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull llama3.1:8b

In [ ]:
%%writefile helpers.py
import argparse
import json
from pathlib import Path

import pandas as pd


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--k", type=int, default=10)
    parser.add_argument("--split", default="test", choices=["dev", "test"])
    parser.add_argument("--text", default="full", choices=["full", "summary"])
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--gold", choices=["names", "descriptions"])  # LLM only: the true labels as categories
    parser.add_argument("--find", default="sample", choices=["sample", "running"])  # LLM only: 400 conversations, or all of train with a running list
    return parser.parse_args()


def load_texts(split, text):
    data = pd.read_csv("data/conversations.csv", keep_default_na=False)
    if text == "summary":
        data = data.merge(pd.read_csv("data/summaries.csv"), on="convo_id")
    data = data[data["split"] == split]
    return data["convo_id"].tolist(), data[text].tolist()


def save_results(method, args, ids, clusters, info):
    name = f"{method}_k{args.k}_{args.split}_{args.text}_s{args.seed}"
    Path("results/predictions").mkdir(parents=True, exist_ok=True)
    Path("results/info").mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"convo_id": ids, "cluster": clusters}).to_csv(f"results/predictions/{name}.csv", index=False)
    with open(f"results/info/{name}.json", "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    print("saved", name)


In [ ]:
%%writefile step1_preprocess.py
import gzip
import json
import urllib.request
from pathlib import Path

import pandas as pd

URL = "https://github.com/asappresearch/abcd/raw/master/data/abcd_v1.1.json.gz"
RAW_FILE = "data/abcd_v1.1.json.gz"


def get_subflow(flow, subflow):
    # the data splits the FAQ subflows into questions (boots_how_3 -> boots), giving 96 instead of 55
    if flow in ["single_item_query", "storewide_query"]:
        return subflow.split("_")[0]
    # only 3 conversations, not a subflow in the ABCD guidelines
    if subflow == "status_delivery_date":
        return "status_delivery_time"
    return subflow


Path("data").mkdir(exist_ok=True)
if not Path(RAW_FILE).exists():
    urllib.request.urlretrieve(URL, RAW_FILE)

with gzip.open(RAW_FILE, "rt", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for split in ["train", "dev", "test"]:
    for convo in raw[split]:
        flow = convo["scenario"]["flow"]
        subflow = convo["scenario"]["subflow"]

        # action turns are system messages that reveal the label, so they are left out
        full = [f"{speaker}: {text}" for speaker, text in convo["original"] if speaker != "action"]

        rows.append({
            "split": split,
            "convo_id": convo["convo_id"],
            "flow": flow,
            "subflow": get_subflow(flow, subflow),
            "full": "\n".join(full),
        })

data = pd.DataFrame(rows)
data.to_csv("data/conversations.csv", index=False)

# same data, but easy to read
with open("data/conversations.txt", "w", encoding="utf-8") as f:
    for _, row in data.sort_values(["flow", "subflow"]).iterrows():
        f.write(f"=== {row['convo_id']} | {row['split']} | {row['flow']} / {row['subflow']} ===\n")
        f.write(row["full"] + "\n\n")

print(data["split"].value_counts())
print("flows:", data["flow"].nunique(), "subflows:", data["subflow"].nunique())


In [ ]:
%%writefile step3_llm.py
import json
import random
import urllib.request
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm import tqdm

from helpers import get_args, load_texts, save_results

MODEL = "llama3.1:8b"
CONTEXT_LENGTH = 8192
CACHE_FILE = "data/llm_answers.jsonl"
GUIDELINES_URL = "https://github.com/asappresearch/abcd/raw/master/data/guidelines.json"
GUIDELINES_FILE = "data/guidelines.json"

SYSTEM = "You analyze customer service conversations for an online clothing store."

FIND_PROMPT = """Below are {n} customer service conversations.

{conversations}

List the distinct reasons customers contacted support in these conversations.
Use short snake_case names. Answer in JSON:
{{"categories": [{{"name": "...", "description": "..."}}]}}"""

MERGE_PROMPT = """Here are customer intent categories:

{categories}

Merge similar categories so that exactly {k} distinct categories remain, numbered 1 to {k}.
Put the most common categories first. Use short snake_case names and a one-sentence description.
Answer in JSON:
{{"categories": [{{"number": 1, "name": "...", "description": "..."}}]}}"""

RUNNING_PROMPT = """Categories found so far:
{categories}

Below are {n} customer service conversations.

{conversations}

For each conversation, give the reason the customer contacted support.
Use a category from the list when one fits. Only make a new short snake_case name if none fits, and then add a one-sentence description.
Answer in JSON, one entry per conversation:
{{"conversations": [{{"name": "...", "description": "..."}}]}}"""

ASSIGN_PROMPT ="""Categories:
{categories}

Conversation:
{conversation}

Which category best describes why the customer contacted support?
Answer in JSON: {{"number": <category number>}}"""

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
tokens = {"prompt": 0, "answer": 0}

# answers are saved, so a stopped run continues where it was
cache = {}
if Path(CACHE_FILE).exists():
    with open(CACHE_FILE, encoding="utf-8") as f:
        for line in f:
            saved = json.loads(line)
            cache[saved["prompt"]] = saved


def ask(prompt, temperature=0.0, max_tokens=1024):
    if prompt not in cache:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
        )
        saved = {
            "prompt": prompt,
            "answer": response.choices[0].message.content,
            "prompt_tokens": response.usage.prompt_tokens,
            "answer_tokens": response.usage.completion_tokens,
        }
        cache[prompt] = saved
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(saved) + "\n")

    saved = cache[prompt]
    tokens["prompt"] += saved["prompt_tokens"]
    tokens["answer"] += saved["answer_tokens"]
    if saved["prompt_tokens"] + max_tokens > CONTEXT_LENGTH:
        print("WARNING: prompt may be cut off,", saved["prompt_tokens"], "tokens")

    try:
        answer = json.loads(saved["answer"])
    except json.JSONDecodeError:
        return {}
    return answer if isinstance(answer, dict) else {}


def valid_categories(answer):
    # skips categories the model wrote in the wrong format, so one bad answer does not stop the run
    categories = answer.get("categories")
    if not isinstance(categories, list):
        return []
    return [c for c in categories if isinstance(c, dict) and isinstance(c.get("name"), str)]


def format_categories(categories):
    lines = []
    for i, category in enumerate(categories, start=1):
        lines.append(f"{i}. {category['name']}: {category.get('description', '')}")
    return "\n".join(lines)


def find_categories(texts, seed, n_batches=20, batch_size=20):
    rng = random.Random(seed)
    found = {}
    for _ in tqdm(range(n_batches), desc="find"):
        batch = rng.sample(texts, batch_size)
        conversations = "\n\n".join(f"[{i + 1}]\n{text[:1000]}" for i, text in enumerate(batch))
        answer = ask(FIND_PROMPT.format(n=batch_size, conversations=conversations), temperature=0.7)
        for category in valid_categories(answer):
            found[category["name"].lower()] = category
    return list(found.values())


def find_categories_running(texts, seed, batch_size=20, max_categories=200):
    # TopicGPT-style: goes through all of train and keeps a list that grows, with a count per category
    texts = random.Random(seed).sample(texts, len(texts))
    found = {}
    for start in tqdm(range(0, len(texts), batch_size), desc="find"):
        batch = texts[start:start + batch_size]
        # only the most used names are shown, so the prompt fits in the context window
        names = sorted(found, key=lambda name: -found[name]["count"])[:max_categories]
        conversations = "\n\n".join(f"[{i + 1}]\n{text[:1000]}" for i, text in enumerate(batch))
        prompt = RUNNING_PROMPT.format(categories="\n".join(names), n=len(batch), conversations=conversations)
        answer = ask(prompt, temperature=0.7, max_tokens=1500)
        for category in valid_categories({"categories": answer.get("conversations")}):
            name = category["name"].lower()
            if name not in found:
                found[name] = {"name": name, "description": category.get("description", ""), "count": 0}
            found[name]["count"] += 1
    found = sorted(found.values(), key=lambda category: -category["count"])[:max_categories]
    # the merge prompt shows the counts, so the model knows which categories are big
    return [{"name": c["name"], "description": f"{c['description']} (used {c['count']} times)"} for c in found]


def merge_categories(categories, k):
    # the model is bad at counting, so its own answer is merged again until it is small enough
    for round_number in range(1, 6):
        prompt = MERGE_PROMPT.format(categories=format_categories(categories), k=k)
        categories = valid_categories(ask(prompt, max_tokens=3000))
        print(f"merge round {round_number}: {len(categories)} categories")
        if len(categories) <= k:
            break
    return categories[:k]


def assign_category(text, categories):
    prompt = ASSIGN_PROMPT.format(categories=format_categories(categories), conversation=text[:4000])
    number = ask(prompt, max_tokens=64).get("number")
    if isinstance(number, int) and 1 <= number <= len(categories):
        return categories[number - 1]["name"]
    return "unassigned"


def flow_descriptions():
    # ABCD's guidelines have a one-line description per flow, e.g. "Product Defect": "refunds and returns"
    if not Path(GUIDELINES_FILE).exists():
        urllib.request.urlretrieve(GUIDELINES_URL, GUIDELINES_FILE)
    with open(GUIDELINES_FILE, encoding="utf-8") as f:
        guidelines = json.load(f)
    return {name.lower().replace(" ", "_").replace("-", "_"): flow["description"] for name, flow in guidelines.items()}


args = get_args()
_, train_texts = load_texts("train", args.text)
ids, texts = load_texts(args.split, args.text)

if args.gold:
    # not unsupervised: shows how well the LLM assigns when the categories are the true ones
    data = pd.read_csv("data/conversations.csv")
    # subflow names like "boots" only make sense with their flow, so they are written as flow/subflow
    names = data["flow"] if args.k == 10 else data["flow"] + "/" + data["subflow"]
    found = []
    categories = [{"name": name} for name in sorted(names.unique())]
    if args.gold == "descriptions":
        # only flows have short descriptions; subflows only have long agent instructions
        assert args.k == 10, "descriptions only exist for flows (k 10)"
        descriptions = flow_descriptions()
        for category in categories:
            category["description"] = descriptions[category["name"]]
else:
    # categories come from train, conversations in dev or test are assigned to them
    if args.find == "running":
        found = find_categories_running(train_texts, args.seed)
    else:
        found = find_categories(train_texts, args.seed)
    categories = merge_categories(found, args.k)
clusters = [assign_category(text, categories) for text in tqdm(texts, desc="assign")]

info = {
    "model": MODEL,
    "found": found,
    "categories": categories,
    "n_categories": len(categories),
    "n_unassigned": clusters.count("unassigned"),
    "tokens": tokens,
}
print("categories:", len(categories), "unassigned:", clusters.count("unassigned"))
method = {None: "llm", "names": "llmgold", "descriptions": "llmgolddesc"}[args.gold]
if args.find == "running":
    method = "llmrun"
save_results(method, args, ids, clusters, info)


In [ ]:
%%writefile step4_evaluate.py
import re
from glob import glob
from pathlib import Path

import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix


def clustering_accuracy(labels, clusters):
    # match each cluster to one label so that as many conversations as possible are right
    counts = contingency_matrix(labels, clusters)
    rows, columns = linear_sum_assignment(counts, maximize=True)
    return counts[rows, columns].sum() / counts.sum()


data = pd.read_csv("data/conversations.csv", usecols=["convo_id", "flow", "subflow"])

rows = []
for file in glob("results/predictions/*.csv"):
    run = Path(file).stem
    predictions = pd.read_csv(file).merge(data, on="convo_id")
    clusters = predictions["cluster"].astype(str)
    for level in ["flow", "subflow"]:
        labels = predictions[level]
        rows.append({
            "setup": re.sub(r"_s\d+$", "", run),
            "level": level,
            "ARI": adjusted_rand_score(labels, clusters),
            "AMI": adjusted_mutual_info_score(labels, clusters),
            "NMI": normalized_mutual_info_score(labels, clusters),
            "ACC": clustering_accuracy(labels, clusters),
            "clusters": clusters.nunique(),
        })

scores = pd.DataFrame(rows)
summary = scores.groupby(["setup", "level"]).agg(
    ARI=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    AMI=("AMI", "mean"),
    NMI=("NMI", "mean"),
    ACC=("ACC", "mean"),
    ACC_std=("ACC", "std"),
    clusters=("clusters", "mean"),
    seeds=("ARI", "size"),
).round(3)

summary.to_csv("results/summary.csv")
print(summary.to_string())


In [ ]:
!python step1_preprocess.py

In [ ]:
# add with Add Input: the summarize notebook (summaries.csv) and, if you have it, an earlier run of this one (llm_answers.jsonl)
!find /kaggle/input -name summaries.csv -exec cp {} data/ \;
!find /kaggle/input -name llm_answers.jsonl -exec cp {} data/ \;
!ls -la data

In [ ]:
# main LLM method: categories from 400 train conversations. dev with 1 seed, test with 3
for text in ["full", "summary"]:
    for k in [10, 55]:
        !python step3_llm.py --k {k} --split dev --text {text}
        for seed in [0, 1, 2]:
            !python step3_llm.py --k {k} --split test --text {text} --seed {seed}

In [ ]:
# true labels as categories (not unsupervised, for comparison)
for text in ["full", "summary"]:
    for k in [10, 55]:
        !python step3_llm.py --k {k} --split test --text {text} --gold names
    !python step3_llm.py --k 10 --split test --text {text} --gold descriptions

In [ ]:
# running category list over all of train (TopicGPT-style), summaries only
for k in [10, 55]:
    !python step3_llm.py --k {k} --split dev --text summary --find running
    for seed in [0, 1, 2]:
        !python step3_llm.py --k {k} --split test --text summary --seed {seed} --find running

In [ ]:
!python step4_evaluate.py
!zip -qr out.zip results data/llm_answers.jsonl